# E(n)-equivariant graph neural network for particle dynamics

*mechanics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/egnn_dynamics)

A tiny E(n)-equivariant graph neural network (EGNN) learns to step a 3D system of charged particles forward in time. Its messages depend only on the distances $|\mathbf x_i - \mathbf x_j|^2$ and its coordinate updates point along $\mathbf x_i - \mathbf x_j$, so rotating the input rotates the prediction exactly; a plain MLP of the same data has to learn this and does not.

**Vector form**

$$\ddot{\mathbf x}_i = \sum_{j} \Big(\frac{c_i c_j}{(\lVert\mathbf x_i - \mathbf x_j\rVert^2 + \varepsilon^2)^{3/2}} - k_s\Big)(\mathbf x_i - \mathbf x_j) \qquad \mathbf v_i \leftarrow \phi_v(\mathbf h_i)\,\mathbf v_i + \frac{1}{N-1}\sum_{j\neq i}(\mathbf x_i - \mathbf x_j)\,\phi_x(\mathbf m_{ij}),\quad \mathbf x_i \leftarrow \mathbf x_i + \tau\,\mathbf v_i$$

**Index (tensor) form**

$$r_{ija} = x_{ia} - x_{ja},\quad d_{ij} = r_{ija}r_{ija} \qquad m_{ijh} = \phi_e(h_i, h_j, d_{ij}, c_ic_j)_h \qquad v_{ia} \leftarrow \phi_v(h_i)\,v_{ia} + \tfrac{1}{N-1}\,r_{ija}\,\phi_x(m_{ij}) \qquad h_{ih} \leftarrow h_{ih} + \phi_h(h_i, m_{ijh})_h$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bia},\ v_{bia}$ | `y[:, 0], y[:, 1]` | (B, N, 3) | position and velocity of particle $i$ in system $b$, Cartesian component $a$ |
| $r_{bija}$ | `x[:, :, None, :] - x[:, None, :, :]` | (B, N, N, 3) | all pair separations by broadcasting |
| $d_{bij} = r_{bija}r_{bija}$ | `einsum('bija,bija->bij', r, r)` | (B, N, N) | repeated $a$ ⇒ summed: squared distances, the only geometric input of the MLPs (rotation invariant) |
| $w_{bij}\,r_{bija}$ | `einsum('bij,bija->bia', w, r)` | (B, N, 3) | true force: repeated $j$ ⇒ summed over partners (Coulomb plus springs) |
| $m_{bijh}$ | `mlp(cat([h_i, h_j, d, q]), phi_e)` | (B, N, N, H) | message on every pair, $H = 32$ features |
| $r_{bija}\,\phi_x(m_{bij})$ | `einsum('bija,bij->bia', r, weight)` | (B, N, 3) | coordinate update: repeated $j$ ⇒ summed; a vector built only from $\mathbf x_i - \mathbf x_j$, so it rotates with the input |
| $\phi_v(h_i)\,v_{ia}$ | `mlp(h, phi_v) * v` | (B, N, 3) | an invariant scalar times a vector: still equivariant |
| $R_{ae}\,x_{bie}$ | `einsum('ae,bie->bia', R, x)` | (B, N, 3) | random rotation used for the equivariance check |

The true system: $N$ unit-mass particles with charges $c_i = \pm 1$, softened Coulomb forces and weak springs $k_s$ between every pair (so the cloud stays together), integrated with RK4 at a step of $\Delta t / 10$. Both networks see the same pairs (state at $t$, state at $t + \Delta t$) from a few random systems and are trained with Adam on the mean squared error; held-out systems give the test error. The EGNN (Satorras, Hoogeboom & Welling, "E(n) Equivariant Graph Neural Networks", ICML 2021) builds a message $m_{ij}$ for every pair from invariant inputs only (node features, $d_{ij} = |\mathbf x_i - \mathbf x_j|^2$ and the charge product), then moves each particle along the directions $\mathbf x_i - \mathbf x_j$ with learned scalar weights. Notice that this is exactly the form of the true force law, $\ddot{\mathbf x}_i = \sum_j w_{ij}(\mathbf x_i - \mathbf x_j)$: the symmetry is built into the architecture instead of being learned from data. The check rotates and shifts a test input by a random $R$ and $\mathbf s$: for the EGNN $f(R\mathbf x + \mathbf s) = Rf(\mathbf x) + \mathbf s$ holds to float32 round-off (about $10^{-6}$), while for the MLP, which sees the flattened coordinates, the mismatch is of the order of the prediction itself. With few training systems the EGNN reaches a lower test error with far fewer weights. In the rollout each model is fed its own predictions; errors compound in both, because the charged system is chaotic, but the EGNN stays close for longer. Simplifications: three layers of width 32, the velocity variant of the EGNN with $\tau = \Delta t / L$ (our choice), full-batch training and a fixed seed. The same equivariant message-passing idea underlies machine-learned interatomic potentials such as NequIP (Batzner et al., Nature Communications 2022) and equivariant diffusion models that generate 3D molecules (Hoogeboom et al., ICML 2022). Try fewer training systems: the gap between the two networks widens.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/egnn_dynamics/main.py
"""Main loop: simulate charged particles → train an EGNN and a plain MLP → check equivariance → roll out → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors

K_SPRING = 0.3  # weak springs between every pair keep the cloud together
EPS = 0.3  # softening of the Coulomb force at short range
N_STEPS = 25  # time steps per training trajectory
N_TEST = 8  # held-out trajectories
HIDDEN = 32  # EGNN feature width
N_LAYERS = 3
MLP_WIDTH = 128
SEED = 0


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    gen = torch.Generator().manual_seed(SEED)
    N, dt = params["n_particles"], params["dt"]
    n_iter = 60 if fast_mode() else params["n_iter"]

    # 1. Data: RK4 trajectories of random systems, cut into (state, next state) pairs.
    x0, v0, c = tensors.initial_states(params["n_train"] + N_TEST, N, gen, dev)
    traj = solve.simulate(x0, v0, c, K_SPRING, EPS, dt, N_STEPS)  # (T + 1, B, 2, N, 3)
    n_tr = params["n_train"]
    train_set = solve.pairs(traj[:, :n_tr], c[:n_tr])
    test_set = solve.pairs(traj[:, n_tr:], c[n_tr:])
    now, nxt, _ = test_set
    drift = ((now[:, 0] + dt * now[:, 1] - nxt[:, 0]) ** 2).mean() + ((now[:, 1] - nxt[:, 1]) ** 2).mean()
    print(f"{n_tr} training systems × {N_STEPS} steps = {len(train_set[0])} pairs, {N} particles in 3D, dt = {dt}")
    print(f"reference: 'no forces' guess (x + dt·v, v) has test MSE {drift.item():.2e}")

    # 2. Train both models on the same pairs.
    net_egnn = tensors.egnn_weights(HIDDEN, N_LAYERS, gen, dev)
    net_mlp = tensors.mlp_weights([7 * N, MLP_WIDTH, MLP_WIDTH, 6 * N], gen, dev)
    results = {}
    for name, net in (("egnn", net_egnn), ("mlp", net_mlp)):
        n_params = sum(p.numel() for p in tensors.parameters(net))
        hist = solve.train(name, net, tensors.parameters(net), train_set, test_set, dt, n_iter, params["lr"])
        results[name] = hist
        print(f"{name.upper():4s} {n_params:6d} weights   train MSE {hist['train_final']:.2e}   "
              f"test MSE {hist['test_final']:.2e}")  # fmt: skip

    # 3. Rotation check on held-out states: rotate + translate the input, compare with rotating the output.
    R, shift = tensors.random_rotation(gen, dev), torch.tensor([0.7, -0.3, 1.1], device=dev)
    for name, net in (("egnn", net_egnn), ("mlp", net_mlp)):
        err = solve.equivariance_error(name, net, now[:, 0], now[:, 1], test_set[2], R, shift, dt)
        print(f"{name.upper():4s} equivariance error |f(Rx+s) - (R f(x)+s)| / |f| = {err:.1e}")

    # 4. Roll out one held-out system for longer than the training trajectories.
    n_roll = params["rollout_steps"]
    b = n_tr  # first test system
    true = solve.simulate(x0[b : b + 1], v0[b : b + 1], c[b : b + 1], K_SPRING, EPS, dt, n_roll)[:, 0, 0]  # (T, N, 3)
    pred = {
        name: solve.rollout(name, net, x0[b : b + 1], v0[b : b + 1], c[b : b + 1], dt, n_roll)[:, 0]
        for name, net in (("egnn", net_egnn), ("mlp", net_mlp))
    }
    for name, xs in pred.items():
        err = (xs - true).norm(dim=-1).mean(dim=-1)  # mean particle position error at each step
        print(f"{name.upper():4s} rollout position error after {n_roll} steps: {err[-1].item():.3f}")

    return plot.animate(true, pred, c[b], results, dt, out_dir / "egnn_dynamics.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/egnn_dynamics/equation.py
r"""Physics → tensor translation: the true force law, an E(n)-equivariant GNN and a plain MLP.

True dynamics. N charged particles (unit mass) held together by weak springs between every pair:

    \ddot x_{bia} = \sum_j w_{bij} (x_{bia} - x_{bja}),
    w_{bij} = c_{bi} c_{bj} (d_{bij} + \varepsilon^2)^{-3/2} - k_s,     d_{bij} = r_{bija} r_{bija}

b = system in the batch, i, j = particles, a = Cartesian component. Repeated a (in d) and j are summed.

EGNN layer (Satorras, Hoogeboom & Welling 2021, the variant with velocities):

    m_{bijh} = \phi_e(h_{bi}, h_{bj}, d_{bij}, q_{bij})                      message on every pair (i ≠ j)
    v_{bia} \leftarrow \phi_v(h_{bi}) v_{bia} + \frac{1}{N-1} \sum_j r_{bija} \phi_x(m_{bij})
    x_{bia} \leftarrow x_{bia} + \tau v_{bia}
    h_{bih} \leftarrow h_{bih} + \phi_h(h_{bi}, \sum_j m_{bijh})

Only the invariant d_{bij} = |x_i - x_j|^2 enters the MLPs φ, and vectors are built only from r_{bija} = x_i - x_j
and v_{bia}. That is why rotating (and translating) the input rotates (and translates) the output exactly.
"""

import torch

Weights = list[tuple[torch.Tensor, torch.Tensor]]  # [(W (out, in), b (out,)), ...] for one small MLP


# --- true physics -----------------------------------------------------------------------------------


def separations(x: torch.Tensor) -> torch.Tensor:
    """r_{bija} = x_{bia} - x_{bja}: (B, N, 3) → (B, N, N, 3), all pairs by broadcasting."""
    return x[:, :, None, :] - x[:, None, :, :]


def squared_distances(r: torch.Tensor) -> torch.Tensor:
    """d_{bij} = r_{bija} r_{bija}  (repeated a ⇒ summed): (B, N, N, 3) → (B, N, N)."""
    return torch.einsum("bija,bija->bij", r, r)


def acceleration(x: torch.Tensor, c: torch.Tensor, k_spring: float, eps: float) -> torch.Tensor:
    r"""\ddot x_{bia} = \sum_j w_{bij} r_{bija}.   x: (B, N, 3), charges c: (B, N) → (B, N, 3)."""
    r = separations(x)
    d = squared_distances(r)
    coulomb = torch.einsum("bi,bj->bij", c, c) * (d + eps**2) ** -1.5  # like charges push apart (w > 0)
    w = coulomb - k_spring  # the springs pull every pair together; the i = j term has r = 0 and drops out
    return torch.einsum("bij,bija->bia", w, r)


def rhs(y: torch.Tensor, c: torch.Tensor, k_spring: float, eps: float) -> torch.Tensor:
    """dy/dt for the state y (B, 2, N, 3): y[:, 0] = positions, y[:, 1] = velocities."""
    return torch.stack([y[:, 1], acceleration(y[:, 0], c, k_spring, eps)], dim=1)


# --- neural networks ---------------------------------------------------------------------------------


def linear(h: torch.Tensor, W: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """y_{...o} = W_{oi} h_{...i} + b_o  (repeated i ⇒ summed), on any number of leading batch indices."""
    return torch.einsum("oi,...i->...o", W, h) + b


def mlp(h: torch.Tensor, weights: Weights) -> torch.Tensor:
    """Linear → SiLU → … → Linear."""
    for W, b in weights[:-1]:
        h = torch.nn.functional.silu(linear(h, W, b))
    W, b = weights[-1]
    return linear(h, W, b)


def egnn_layer(
    h: torch.Tensor, x: torch.Tensor, v: torch.Tensor, q: torch.Tensor, layer: dict, tau: float
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """One equivariant layer. h (B, N, H), x and v (B, N, 3), q_{bij} = c_i c_j (B, N, N)."""
    B, N, H = h.shape
    r = separations(x)  # (B, N, N, 3)
    d = squared_distances(r)  # (B, N, N)   invariant
    pair = torch.cat(
        [h[:, :, None, :].expand(B, N, N, H), h[:, None, :, :].expand(B, N, N, H), d[..., None], q[..., None]], dim=-1
    )  # (B, N, N, 2H + 2)
    not_self = 1.0 - torch.eye(N, device=x.device)  # (N, N): no message from a particle to itself
    m = mlp(pair, layer["phi_e"]) * not_self[None, :, :, None]  # m_{bijh}   (B, N, N, H)

    weight = mlp(m, layer["phi_x"])[..., 0]  # φ_x(m_{bij})   (B, N, N)   a scalar per pair
    push = torch.einsum("bija,bij->bia", r, weight) / (N - 1)  # Σ_j r_{bija} φ_x(m_{bij})   (B, N, 3)
    v = mlp(h, layer["phi_v"]) * v + push  # φ_v(h_i) is a scalar per particle: (B, N, 1) * (B, N, 3)
    x = x + tau * v

    m_sum = m.sum(dim=2)  # Σ_j m_{bijh}   (B, N, H)
    h = h + mlp(torch.cat([h, m_sum], dim=-1), layer["phi_h"])
    return h, x, v


def egnn(x: torch.Tensor, v: torch.Tensor, c: torch.Tensor, net: dict, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Predict (x, v) one time step dt later. The L layers each move the particles by dt / L."""
    speed2 = torch.einsum("bia,bia->bi", v, v)  # |v_i|², invariant
    h = mlp(torch.stack([c, speed2], dim=-1), net["embed"])  # (B, N, H) node features
    q = torch.einsum("bi,bj->bij", c, c)  # edge attribute: product of charges
    tau = dt / len(net["layers"])
    for layer in net["layers"]:
        h, x, v = egnn_layer(h, x, v, q, layer, tau)
    return x, v


def plain_mlp(
    x: torch.Tensor, v: torch.Tensor, c: torch.Tensor, weights: Weights, dt: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Baseline: flatten every coordinate into one vector and let an MLP predict the change. Not equivariant."""
    B, N, _ = x.shape
    flat = torch.cat([x.reshape(B, -1), v.reshape(B, -1), c], dim=-1)  # (B, 7N)
    delta = mlp(flat, weights).reshape(B, 2, N, 3)
    return x + dt * v + delta[:, 0], v + delta[:, 1]  # same "drift" head start the EGNN gets from x ← x + τv


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/egnn_dynamics/tensors.py
"""Build the concrete tensors: random initial states, charges, network weights and a random rotation."""

import math

import torch


def initial_states(n_systems: int, n_particles: int, gen: torch.Generator, device: torch.device):
    """Random systems: positions x (B, N, 3), velocities v (B, N, 3) and charges c (B, N) in {-1, +1}.

    Positions and velocities are isotropic Gaussians, so the data set has no preferred direction.
    The centre of mass is put at rest at the origin.
    """
    x = torch.randn(n_systems, n_particles, 3, generator=gen)
    v = 0.5 * torch.randn(n_systems, n_particles, 3, generator=gen)
    x = x - x.mean(dim=1, keepdim=True)
    v = v - v.mean(dim=1, keepdim=True)
    c = torch.randint(0, 2, (n_systems, n_particles), generator=gen).float() * 2 - 1
    return x.to(device), v.to(device), c.to(device)


def mlp_weights(sizes: list[int], gen: torch.Generator, device: torch.device, last_bias: float = 0.0):
    """[(W (out, in), b (out,)), ...] with the usual 1/√in scale. All tensors require gradients."""
    weights = []
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        W = torch.randn(n_out, n_in, generator=gen) / math.sqrt(n_in)
        b = torch.zeros(n_out)
        if i == len(sizes) - 2:
            W = 0.1 * W  # start close to "no correction"
            b = b + last_bias
        weights.append((W.to(device).requires_grad_(), b.to(device).requires_grad_()))
    return weights


def egnn_weights(hidden: int, n_layers: int, gen: torch.Generator, device: torch.device) -> dict:
    """Weights of the EGNN: an embedding of (c_i, |v_i|²) and n_layers × (φ_e, φ_x, φ_v, φ_h)."""
    H = hidden
    layers = [
        {
            "phi_e": mlp_weights([2 * H + 2, H, H], gen, device),
            "phi_x": mlp_weights([H, H, 1], gen, device),
            "phi_v": mlp_weights([H, H, 1], gen, device, last_bias=1.0),  # φ_v ≈ 1: keep the velocity at first
            "phi_h": mlp_weights([2 * H, H, H], gen, device),
        }
        for _ in range(n_layers)
    ]
    return {"embed": mlp_weights([2, H], gen, device), "layers": layers}


def parameters(net) -> list[torch.Tensor]:
    """Flatten the nested weights (dict / list / tuple) into the list the optimiser needs."""
    if isinstance(net, torch.Tensor):
        return [net]
    items = net.values() if isinstance(net, dict) else net
    return [p for item in items for p in parameters(item)]


def random_rotation(gen: torch.Generator, device: torch.device) -> torch.Tensor:
    """A random proper rotation R_{ab} (3, 3) with det R = +1, from the QR decomposition of a Gaussian matrix."""
    Q, R = torch.linalg.qr(torch.randn(3, 3, generator=gen, dtype=torch.float64))
    Q = Q * torch.sign(torch.diagonal(R))  # make the decomposition unique
    if torch.det(Q) < 0:
        Q[:, 0] = -Q[:, 0]
    return Q.float().to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/egnn_dynamics/solve.py
"""Simulate the true system with RK4, train the EGNN and the MLP on (state → next state), roll them out."""

import torch

from problems._common.integrators import integrate
from problems._common.progress import steps

from .equation import egnn, plain_mlp, rhs


def simulate(x0, v0, c, k_spring: float, eps: float, dt: float, n_steps: int, substeps: int = 10) -> torch.Tensor:
    """True trajectories sampled every dt: (n_steps + 1, B, 2, N, 3). RK4 runs with the finer step dt / substeps."""
    y0 = torch.stack([x0, v0], dim=1)  # (B, 2, N, 3)
    traj = integrate(lambda y, t: rhs(y, c, k_spring, eps), y0, dt / substeps, n_steps * substeps)
    return traj[::substeps]


def pairs(traj: torch.Tensor, c: torch.Tensor):
    """Every (state at t, state at t + dt) of every trajectory, flattened into one batch of P = T·B pairs."""
    T1, B = traj.shape[:2]
    now = traj[:-1].reshape(-1, *traj.shape[2:])  # (P, 2, N, 3)
    nxt = traj[1:].reshape(-1, *traj.shape[2:])
    charges = c[None].expand(T1 - 1, B, -1).reshape(-1, c.shape[-1])  # (P, N)
    return now, nxt, charges


def predict(model: str, net, x, v, c, dt: float):
    return egnn(x, v, c, net, dt) if model == "egnn" else plain_mlp(x, v, c, net, dt)


def mse(model: str, net, now, nxt, c, dt: float) -> torch.Tensor:
    """Mean squared error of the predicted next positions and velocities."""
    x, v = predict(model, net, now[:, 0], now[:, 1], c, dt)
    return ((x - nxt[:, 0]) ** 2).mean() + ((v - nxt[:, 1]) ** 2).mean()


def train(model: str, net, params: list, train_set, test_set, dt: float, n_iter: int, lr: float) -> dict:
    """Full-batch Adam with a cosine learning-rate decay. Returns the train and test loss histories."""
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_iter, eta_min=lr / 30)
    history = {"train": [], "test": []}
    for _ in steps(n_iter, f"training {model}"):
        loss = mse(model, net, *train_set, dt)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        history["train"].append(loss.item())
    with torch.no_grad():
        history["test_final"] = mse(model, net, *test_set, dt).item()
        history["train_final"] = mse(model, net, *train_set, dt).item()
    return history


@torch.no_grad()
def rollout(model: str, net, x0, v0, c, dt: float, n_steps: int) -> torch.Tensor:
    """Feed the model its own predictions n_steps times: (n_steps + 1, B, N, 3) positions."""
    xs, x, v = [x0], x0, v0
    for _ in range(n_steps):
        x, v = predict(model, net, x, v, c, dt)
        xs.append(x)
    return torch.stack(xs)


@torch.no_grad()
def equivariance_error(model: str, net, x, v, c, R: torch.Tensor, shift: torch.Tensor, dt: float) -> float:
    """max |f(R x + s, R v) - (R f_x + s, R f_v)| / max |f|: zero for an E(3)-equivariant model.

    Rotations act on the Cartesian index:  x'_{bia} = R_{ae} x_{bie} + s_a  (repeated e ⇒ summed).
    """

    def rot(u: torch.Tensor) -> torch.Tensor:
        return torch.einsum("ae,bie->bia", R, u)

    x_out, v_out = predict(model, net, x, v, c, dt)
    x_rot, v_rot = predict(model, net, rot(x) + shift, rot(v), c, dt)
    err = torch.maximum((x_rot - (rot(x_out) + shift)).abs().max(), (v_rot - rot(v_out)).abs().max())
    scale = torch.maximum(x_out.abs().max(), v_out.abs().max())
    return (err / scale).item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/egnn_dynamics/plot.py
"""Animate the rollout (true vs EGNN vs MLP, seen from above) with the rollout error and the training curves."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

GREY, BLUE, ORANGE, RED = "#6b7280", "#2563eb", "#ea580c", "#dc2626"


def animate(true: torch.Tensor, pred: dict, charges: torch.Tensor, results: dict, dt: float, path: Path) -> Path:
    tr = true.cpu().numpy()  # (T, N, 3)
    eg, ml = pred["egnn"].cpu().numpy(), pred["mlp"].cpu().numpy()
    T = len(tr)
    t = np.arange(T) * dt
    err_eg = np.linalg.norm(eg - tr, axis=-1).mean(-1)  # (T,)
    err_ml = np.linalg.norm(ml - tr, axis=-1).mean(-1)
    q = charges.cpu().numpy()

    fig = plt.figure(figsize=(8.4, 4.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.25, 1])
    ax = fig.add_subplot(grid[:, 0])
    ax_e = fig.add_subplot(grid[0, 1])
    ax_l = fig.add_subplot(grid[1, 1])

    # --- rollout, projected on the x–y plane ---
    lim = np.abs(tr[..., :2]).max() * 1.25
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("rollout (top view): true vs predicted", fontsize=10)
    face = np.where(q > 0, RED, BLUE)
    trails = [ax.plot([], [], color=GREY, lw=1, alpha=0.6)[0] for _ in range(tr.shape[1])]
    dots_true = ax.scatter(
        tr[0, :, 0], tr[0, :, 1], s=70, c=face, edgecolors="#111827", zorder=4, label="true (± charge)"
    )
    dots_eg = ax.scatter(eg[0, :, 0], eg[0, :, 1], s=150, facecolors="none", edgecolors=BLUE, lw=1.8, zorder=5,
                         label="EGNN")  # fmt: skip
    dots_ml = ax.scatter(ml[0, :, 0], ml[0, :, 1], s=60, marker="x", color=ORANGE, lw=1.8, zorder=5, label="MLP")
    ax.legend(loc="lower left", fontsize=7)
    clock = ax.text(0.03, 0.96, "", transform=ax.transAxes, va="top", fontsize=9)

    # --- rollout error ---
    ax_e.set_xlim(0, t[-1])
    floor = max(1e-4, min(err_eg[1:].min(), err_ml[1:].min()) * 0.5) if T > 1 else 1e-4
    ax_e.set_yscale("log")
    ax_e.minorticks_off()
    ax_e.set_ylim(floor, max(err_eg.max(), err_ml.max(), floor) * 2)
    ax_e.set_title("rollout: mean position error", fontsize=10)
    ax_e.tick_params(labelsize=8)
    (line_eg,) = ax_e.plot([], [], color=BLUE, lw=1.8, label="EGNN")
    (line_ml,) = ax_e.plot([], [], color=ORANGE, lw=1.8, label="MLP")
    ax_e.legend(loc="upper left", fontsize=7)

    # --- training curves (static) ---
    for name, color in (("egnn", BLUE), ("mlp", ORANGE)):
        h = np.array(results[name]["train"])
        ax_l.plot(np.arange(1, len(h) + 1), h, color=color, lw=1.2, label=f"{name.upper()} train")
        ax_l.axhline(results[name]["test_final"], color=color, ls="--", lw=1, label=f"{name.upper()} test")
    ax_l.set_yscale("log")
    ax_l.minorticks_off()
    ax_l.set_xlabel("iteration", fontsize=9)
    ax_l.set_title("one-step MSE", fontsize=10)
    ax_l.tick_params(labelsize=8)
    ax_l.legend(loc="upper right", fontsize=6, ncol=2)

    def update(s: int):
        lo = max(0, s - 15)
        for n, trail in enumerate(trails):
            trail.set_data(tr[lo : s + 1, n, 0], tr[lo : s + 1, n, 1])
        dots_true.set_offsets(tr[s, :, :2])
        dots_eg.set_offsets(eg[s, :, :2])
        dots_ml.set_offsets(ml[s, :, :2])
        line_eg.set_data(t[: s + 1], err_eg[: s + 1])
        line_ml.set_data(t[: s + 1], err_ml[: s + 1])
        clock.set_text(f"t = {t[s]:.1f}  (step {s})")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(T, 80), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_particles': 5,
    'n_train': 12,
    'dt': 0.1,
    'n_iter': 600,
    'lr': 0.003,
    'rollout_steps': 50
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.egnn_dynamics")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.egnn_dynamics.main")
problem_dir = Path("problems/mechanics/egnn_dynamics")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
